### Problem Statement 3C

3C. Implement Anomaly Detection in Energy Consumption Data (Smart Grid Dataset) using Autoencoder and build the model by using the following steps:
Steps:
- Preprocess the data to fill any missing values, normalize, and create time windows for sequential input.
- Use an LSTM-based autoencoder to learn temporal patterns in normal consumption.
- Set an error threshold and detect anomalies when the reconstruction error exceeds this threshold.
Dataset: The Smart Grid dataset records electricity consumption across multiple timestamps for various households.
Goal: Detect unusual consumption patterns by training an autoencoder on typical (normal) energy usage patterns

In [ ]:
# Imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, RepeatVector, TimeDistributed, Dense
from sklearn.preprocessing import MinMaxScaler

In [ ]:
# Preprocess data: fill missing values, normalize, and create time windows
csv_path = "../LP IV Updated Datasets/AutoEncoder/Energy Consumption/energy consumption.csv"

data = pd.read_csv(csv_path)
print("Data shape:", data.shape)
print(data.head())

# Fill any missing values
data = data.ffill()

# Normalize numerical features using MinMaxScaler
feature_cols = ['Electricity_Consumed', 'Temperature', 'Humidity', 'Wind_Speed', 'Avg_Past_Consumption']
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(data[feature_cols])

# Create time windows (sequence length = 10 steps)
TIME_STEPS = 10
X_seq = []
y_seq = []

for i in range(len(scaled_data) - TIME_STEPS):
    X_seq.append(scaled_data[i : i + TIME_STEPS])
    y_seq.append(data['Anomaly_Label'].values[i + TIME_STEPS])

X_seq = np.array(X_seq)
y_seq = np.array(y_seq)

# Filter normal sequences (Anomaly_Label == 0) for training
X_train_normal = X_seq[y_seq == 0]

print("Sequences shape:", X_seq.shape)
print("Normal training sequences:", X_train_normal.shape[0])

In [ ]:
# Use an LSTM-based autoencoder to learn temporal patterns
n_features = X_train_normal.shape[2]

model = Sequential()
# Encoder
model.add(LSTM(32, activation='relu', input_shape=(TIME_STEPS, n_features), return_sequences=False))
model.add(RepeatVector(TIME_STEPS))
# Decoder
model.add(LSTM(32, activation='relu', return_sequences=True))
model.add(TimeDistributed(Dense(n_features)))

model.compile(optimizer='adam', loss='mse')
model.summary()

In [ ]:
# Train LSTM autoencoder on normal consumption patterns
history = model.fit(X_train_normal, X_train_normal, epochs=5, batch_size=32, validation_split=0.1)

In [ ]:
# Set an error threshold and detect anomalies
# Predict on all sequences
predictions = model.predict(X_seq)

# Calculate reconstruction error (MSE) for each sequence
mse = np.mean(np.square(X_seq - predictions), axis=(1, 2))

# Set threshold at 95th percentile of normal training errors
threshold = np.percentile(mse[y_seq == 0], 95)
print("Anomaly threshold:", threshold)

# Predict anomalies (error > threshold is an anomaly)
y_pred = (mse > threshold).astype(int)

from sklearn.metrics import classification_report
print("\nClassification Report:")
print(classification_report(y_seq, y_pred, target_names=['Normal', 'Anomaly']))

# Plot reconstruction errors and threshold line
plt.figure(figsize=(10, 4))
plt.plot(mse[:200], label='Reconstruction Error')
plt.axhline(threshold, color='red', linestyle='--', label='Threshold')
plt.title('Reconstruction Errors vs Threshold')
plt.xlabel('Sequence Index')
plt.ylabel('MSE')
plt.legend()
plt.show()